In [24]:
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader('Why_Language_Models_Hallucinate_Explainer.pdf')
# Loader 
pages = loader.load()


In [25]:
from torch import chunk
import chromadb
# Create chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
spliter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=140)
text = spliter.split_documents(pages)
chunks = [i.page_content for i in text]
metadata = [i.metadata for i in text]

# ChromaDB
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')

client = chromadb.PersistentClient(path="./Retrival")
collection = client.get_or_create_collection(name="Simple_DATAbase",embedding_function=embedding_function)

if collection.count()==0 :
    collection.add(
        documents=chunks,
        ids=[str(i) for i in range (len(chunks))],
        metadatas=metadata
    )

from rank_bm25 import BM25Okapi
token_corpus = [c.split() for c in chunks]
bm25 = BM25Okapi(token_corpus)


In [38]:
def Hybrid_tokenizer(query:str):
    result = collection.query(query_texts=[query],n_results=5)
    document = result['documents'][0]
    distance = result['distances'][0]
    thresold = 1.0 
    dense_corpus=[]
    for doc , d in zip (document,distance):
        if d < thresold:

            print(d,doc)
            dense_corpus.append(doc)
    
    scores = bm25.get_scores(query.split())
    print(f"the len of chunks: {len(chunks)}")
    print(len(scores))
    def get_tokens(scores , k=10):
        index = list(enumerate(scores))
        index_sorted = sorted(index,key=lambda x:x[1],reverse=True)
        return [s for s, i in index_sorted[:k]]
    
    close_token=get_tokens(scores,k=10)

    print("len(scores):", len(scores))
    print("k used:", 10)
    print("len(close_token):", len(close_token))
    print(close_token)
    clone_close_token = [chunks[i] for i in close_token]

    rrn_ranker={}
    for rank , doc in enumerate(dense_corpus):
        rrn_ranker[doc]=rrn_ranker.get(doc,0)+1/(60+rank)
    for rank,doc in enumerate(clone_close_token):
        rrn_ranker[doc]=rrn_ranker.get(doc,0)+1/(60+rank)

    marge = sorted(rrn_ranker.items() , key= lambda x : x[1] , reverse=True)
    top_doc =[doc for doc , _ in marge[:5]]

    if not top_doc:
        return "NOT RELEVENT CONTENT"

    return "\n\n".join(top_doc)


print(Hybrid_tokenizer('llm full form?'))


0.8525054454803467 1
 Why Language Models Hallucinate: A Statistical Account
 A Summary and Synthesis of Current Research  |  Compiled 2026
ABSTRACT
Hallucination — the tendency of large language models (LLMs) to produce fluent, confident, and factually
wrong statements — is often described as mysterious or unpredictable. Recent theoretical work, most
notably by Kalai, Nachum, Vempala, and Zhang (2025), argues the opposite: hallucination is a natural and
largely predictable consequence of how models are trained and graded, not an unexplained quirk of scale.
This document synthesizes that argument alongside supporting survey literature, covering the pretraining
origins of factual error, why post-training and benchmarking fail to eliminate it, the specific error categories
that resist correction regardless of model size, and the mitigation strategies — including retrieval-augmented
generation and evaluation reform — currently under study.
1. Framing the Problem
0.8777539730072021 generat

In [39]:
from dotenv import load_dotenv 
import os 
load_dotenv()
key = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq 
cha_model = ChatGroq(model_name = 'llama-3.1-8b-instant' , temperature=0,api_key=key)

qustion = "What is LLM ?"
answers = Hybrid_tokenizer(query=qustion)
prompt = f"""
you are a reliable LLM so , provide me answers based on the content dont use outside content any time , 
content : {answers}
qustion : {qustion}
"""
cha_model.invoke(prompt)


0.838268518447876 1
 Why Language Models Hallucinate: A Statistical Account
 A Summary and Synthesis of Current Research  |  Compiled 2026
ABSTRACT
Hallucination — the tendency of large language models (LLMs) to produce fluent, confident, and factually
wrong statements — is often described as mysterious or unpredictable. Recent theoretical work, most
notably by Kalai, Nachum, Vempala, and Zhang (2025), argues the opposite: hallucination is a natural and
largely predictable consequence of how models are trained and graded, not an unexplained quirk of scale.
This document synthesizes that argument alongside supporting survey literature, covering the pretraining
origins of factual error, why post-training and benchmarking fail to eliminate it, the specific error categories
that resist correction regardless of model size, and the mitigation strategies — including retrieval-augmented
generation and evaluation reform — currently under study.
1. Framing the Problem
0.8639163374900818 - The fa

AIMessage(content='Based on the provided content, a Large Language Model (LLM) is not explicitly defined in the abstract or the references. However, based on the context, it can be inferred that an LLM refers to a type of artificial intelligence model that is trained on large amounts of text data to generate human-like language.\n\nIn the content, LLMs are mentioned as being capable of producing fluent, confident, and factually wrong statements, which is referred to as "hallucination." This suggests that LLMs are complex models that can generate text based on patterns and associations learned from the training data, but may not always be accurate or reliable.\n\nIn the context of the provided content, LLMs are likely referring to models such as those developed by OpenAI, such as GPT-3, which are designed to generate human-like text based on input prompts.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 175, 'prompt_tokens': 1108, 'total_tokens': 1283, 'c